# Day 09: Object-Oriented Programming — Part 2

## Part 1: The `del` Keyword

Used to delete an object's attribute, or the object itself.
```python
del s1.name  # deletes just that attribute
del s1       # deletes the object entirely
```


## Part 2: "Private" Attributes and Methods

Attributes or methods meant to be used only inside the class, not accessed from outside it. Python doesn't have real private/public access control like some other languages — this is more of a strong convention than a hard rule. Prefixing a name with `__` (double underscore) makes it behave like a private attribute.

In [1]:
class Account():
    def __init__(self, number, password):
        self.number = number
        self.__password = password  # the __ prefix makes this "private"

    def reset(self):
        print(self.__password)

acc1 = Account(1234, "rahul")
print(acc1.number)   # regular attribute — accessible from outside
acc1.reset()          # accessible through a method defined inside the class

try:
    print(acc1.password)   # trying to access it directly from outside fails
except AttributeError as e:
    print(f"Can't access it directly: {e}")

1234
rahul
Can't access it directly: 'Account' object has no attribute 'password'


## Part 3: Inheritance

When one class (child/derived) picks up the properties and methods of another class (parent/base).

In [2]:
class Car:
    color = "black"

    @staticmethod
    def start():
        print("Car start..")

    @staticmethod
    def stop():
        print("Car stops..")

class ToyotaCar(Car):
    def __init__(self, name):
        self.name = name

car1 = ToyotaCar("Fortuner")
print(car1.name)
car1.start()
car1.stop()

car2 = ToyotaCar("Prius")  # a separate object, with its own name
print(car2.name)

Fortuner
Car start..
Car stops..
Prius


### Types of Inheritance

1. **Single** — `base` → `derived`
2. **Multilevel** — `base` → `derived` → `derived`
3. **Multiple** — `base1` → `derived` ← `base2`


### Multilevel Inheritance

Building `Fortuner` here (`Car` → `ToyotaCar` → `Fortuner`), I found the class was actually broken — `Fortuner.__init__` only forwarded `brand` to `super().__init__()`, but `ToyotaCar.__init__` needs both `brand` **and** `color`. Trying to actually create a `Fortuner` would have raised a `TypeError` for a missing argument. Fixed it below by passing both through.

I also renamed the `type` parameter to `fuel_type` — `type` is a Python built-in, and using it as a parameter name shadows it for the rest of that function.

In [3]:
class Car:
    def __init__(self, color):
        self.color = color

    @staticmethod
    def start():
        print("Car start..")

    @staticmethod
    def stop():
        print("Car stops..")

class ToyotaCar(Car):
    def __init__(self, brand, color):
        self.brand = brand
        super().__init__(color)  # super() gives access to the parent class

class Ford(Car):
    def __init__(self, brand, color):
        self.brand = brand
        super().__init__(color)

class Fortuner(ToyotaCar):
    def __init__(self, fuel_type, brand, color):
        self.fuel_type = fuel_type
        super().__init__(brand, color)  # fixed — now forwards both required arguments

class Mustang(Ford):
    def __init__(self, fuel_type, brand, color):
        self.fuel_type = fuel_type
        super().__init__(brand, color)

c1 = Mustang("diesel", "Boult_Mustang", "Red")
print(c1.fuel_type)
print(c1.brand)
print(c1.color)
c1.start()
c1.stop()

c2 = Fortuner("petrol", "Toyota_Fortuner", "White")  # now this actually works
print(c2.fuel_type)
print(c2.brand)
print(c2.color)

diesel
Boult_Mustang
Red
Car start..
Car stops..
petrol
Toyota_Fortuner
White


### Multiple Inheritance

In [4]:
class A:
    value1 = "Welcome to class A"

class B:
    value2 = "Welcome to class B"

class C(A, B):
    value3 = "Welcome to class C"

c1 = C()
print(c1.value1)
print(c1.value2)
print(c1.value3)

Welcome to class A
Welcome to class B
Welcome to class C


## Part 4: Class Methods

A class method can access and modify class-level state (data shared by every instance), or call other class methods — without needing an object to exist first. It takes `cls` instead of `self`.

In [7]:
class Student():
    name = "anonymous"

    def set_name(self, name):
        self.name = name  # only changes this one object's own copy, not the shared class value

    def change(self, name, surname):
        Student.name = name                 # directly changes the class attribute
        self.__class__.surname = surname     # same effect, written a different way

    @classmethod
    def change_cls(cls, name, surname):
        # does the same thing as change(), but the proper way — called on the class itself
        cls.name = name
        cls.surname = surname


p1 = Student()
p1.set_name("rahul")
print(f"p1 --> {p1.name}")

p2 = Student()
print(f"p2 --> {p2.name}")  # still "anonymous" — p1's change only affected p1

p3 = Student()
p3.change("rahul", "tathod")
print(f"p3 --> {p3.name} {p3.surname}")

p2 = Student()  # a brand-new p2 object
print(f"p2 --> {p2.name}")  # now shows "rahul" — p3.change() updated the shared class attribute

Student.change_cls("abhijeet", "yadgire")  # called directly on the class, no object needed
p4 = Student()
print(f"p4 --> {p4.name} {p4.surname}")

p2 = Student()
print(f"p2 --> {p2.name}")#now shows "Abhijeet" — p4.change_cls() updated the shared class attribute

p1 --> rahul
p2 --> anonymous
p3 --> rahul tathod
p2 --> rahul
p4 --> abhijeet yadgire
p2 --> abhijeet


## Part 5: The Three Method Types — Recap

1. **Static method** — doesn't need `self` or `cls` at all, since it doesn't touch instance or class data. Called directly on the class.
2. **Class method** — takes `cls`, used to read or modify class-level state shared by every instance.
3. **Instance method** — takes `self`, used to read or modify data that belongs to one specific object.


## Key Takeaways

- `__` prefix creates a "private-like" attribute — a strong convention, not true access control like some other languages enforce.
- `super()` calls up to the parent class — but whatever it calls (usually `__init__`) still needs to receive exactly the arguments it expects. A subclass that only forwards *some* of them will fail the moment it's actually instantiated.
- Inheritance comes in a few shapes: single (one parent), multilevel (a chain of parents), and multiple (more than one direct parent).
- Static methods take neither `self` nor `cls`; class methods take `cls` and affect every instance; instance methods take `self` and affect just one object.
- Avoid parameter names that shadow Python built-ins — `type`, `list`, `str`, `sum`, etc.
